# Week 10 Homework: ARIA v7.0 - The All-Weather Auditor

**Course:** Remote Sensing & Spatial Information Analysis  
**Student:** Wade  
**Case:** Hualien, Typhoon Fung-wong  
**Objective:** integrate Sentinel-1 SAR, Sentinel-2 NDWI/cloud masking, and DEM slope auditing to detect flood candidates under cloud cover.

**Captain's Log:** The assignment expected a local `S1_Hualien_dB.tif`, but it was not available in the provided folder. I therefore used the Week 10 STAC workflow to stream Sentinel-1 RTC from Microsoft Planetary Computer. This is documented because data provenance is part of the audit trail.


## Week 10 課程重點理解

本週課程不是只學會畫 SAR 圖，而是學會在災害情境中判斷「資料是否可信」。Week 8–9 主要使用 Sentinel-2 光學影像與 NDVI/NDWI/BSI 等指標，但颱風期間雲量很高，光學影像常常看不到地表。Week 10 的 ARIA v7.0 因此加入 Sentinel-1 SAR，利用微波可穿透雲層的特性，在光學資料失效時仍提供水體候選區。

課程的第一個重點是 SAR 水體判釋的物理意義：平滑水面會造成鏡面反射，回到衛星的 VV backscatter 較低，所以在 dB 影像上呈現暗色。第二個重點是 SAR 影像有 speckle noise，不能直接 threshold raw SAR；必須先做 median filter，再用 `VV < -18 dB` 或其他門檻萃取候選水體。第三個重點是 sensor fusion：High Confidence 需要 SAR 與非雲區 NDWI 同時支持；若光學被雲遮蔽，SAR 偵測只能標為 SAR Only (Cloudy)，不能假裝成雙重確認。

最後一個重點是 topographic audit。SAR 是側視幾何，山區可能因 radar shadow、layover、foreshortening 產生暗區，這些暗區可能被誤判為水。因此本作業用 DEM slope > 25° 標記 false-positive candidates。這讓 ARIA v7.0 的成果不只是「有跑出圖」，而是能回答：水體候選區從哪裡來、為什麼可信、哪裡需要現地驗證。


## Operation History - How the Results Were Produced

The table below records the actual processing path from data discovery to final report. This is the missing link between the figures and the numbers.

| Stage                | Operation                                                                                  | Result                                                                        |
|:---------------------|:-------------------------------------------------------------------------------------------|:------------------------------------------------------------------------------|
| 1. Data check        | Looked for local S1_Hualien_dB.tif in Week 10 folder                                       | File was not present, so the workflow switched to STAC streaming.             |
| 2. SAR search        | Planetary Computer STAC, collection sentinel-1-rtc, Hualien BBOX, 2025-09-10 to 2025-09-25 | Selected Sentinel-1 VV scene on 2025-09-18, orbit ascending.                  |
| 3. SAR conversion    | Read VV linear backscatter and applied 10 * log10(value)                                   | Created SAR dB raster for threshold-based water extraction.                   |
| 4. Speckle control   | Applied 5 x 5 median filter before thresholding                                            | Reduced isolated dark speckle that can become false water pixels.             |
| 5. SAR water mask    | Applied VV < -18 dB, then morphology and connected-component cleanup                       | Produced 1,038 SAR flood-like pixels, equal to 0.934 km2.                     |
| 6. Optical mask      | Loaded Sentinel-2 L2A green, NIR, and SCL; computed NDWI and cloud mask                    | Selected optical scene is 100.0% cloud-covered, so NDWI cannot confirm water. |
| 7. Fusion            | Combined SAR water, NDWI water, and SCL cloud mask using the Week 10 rule table            | High confidence = 0.000 km2; SAR-only cloudy = 0.934 km2.                     |
| 8. Topographic audit | Loaded Copernicus DEM, calculated slope, flagged flood detections where slope > 25 degrees | Flagged 0.613 km2 as steep-slope false positives.                             |
| 9. Reporting         | Saved figures, CSV tables, AI briefing, and this notebook                                  | All outputs are under week10_outputs/.                                        |

### Processing Chain

```text
Sentinel-1 RTC VV
  -> convert linear backscatter to dB
  -> 5 x 5 median filter
  -> VV < -18 dB threshold
  -> morphological opening and connected-component cleanup
  -> SAR flood-like water mask

Sentinel-2 L2A B03/B08/SCL
  -> NDWI = (Green - NIR) / (Green + NIR)
  -> SCL cloud mask
  -> optical water and cloud evidence

Copernicus DEM
  -> slope calculation
  -> slope > 25 degrees topographic audit

SAR mask + NDWI mask + cloud mask + slope audit
  -> ARIA v7.0 confidence map and final interpretation
```


## Executive Summary

| Metric                                | Value                                                              |
|:--------------------------------------|:-------------------------------------------------------------------|
| SAR acquisition                       | 2025-09-18                                                         |
| SAR source                            | S1A_IW_GRDH_1SDV_20250918T100123_20250918T100148_061041_079B51_rtc |
| SAR orbit                             | ascending                                                          |
| Spatial window                        | BBOX 121.28, 23.56, 121.52, 23.76                                  |
| Working grid                          | 30 m                                                               |
| SAR threshold                         | -18 dB                                                             |
| NDWI threshold                        | 0.0                                                                |
| Cloud cover in selected optical scene | 100.0%                                                             |
| SAR flood-like water pixels           | 1,038                                                              |
| SAR flood-like water area             | 0.934 km2                                                          |
| Mean flood-zone VV backscatter        | -19.62 dB                                                          |
| High-confidence dual-sensor area      | 0.000 km2                                                          |
| SAR-only cloudy area                  | 0.934 km2                                                          |
| Steep-slope false positives flagged   | 0.613 km2                                                          |
| Post-audit flood candidate area       | 0.321 km2                                                          |

**Main finding:** under a 100.0% cloud-covered Sentinel-2 scene, optical confirmation is unavailable. ARIA v7.0 therefore reports **0.934 km2** as SAR-only cloudy flood candidates, then flags **0.613 km2** of steep-slope detections for manual review. The post-audit candidate area is approximately **0.321 km2**.


## Reproducibility Note

The numeric workflow is reproducible with:

```bash
python generate_week10_submission.py
```

The script regenerates the SAR/optical/DEM processing outputs: figures, CSV tables, `.env`, and AI briefing markdown. This submitted notebook then documents the operation history, figure interpretation, and discussion around those generated outputs. I kept `.env` local and uploaded `.env.example` to GitHub to follow the homework instruction not to commit environment files.


## Task 1: SAR All-Weather Flood Detection

### How This Result Was Generated

The SAR layer comes from Sentinel-1 RTC VV. Smooth open water usually returns weak radar energy, so it appears dark in VV dB. The workflow first converts the linear backscatter to dB, then applies a median filter before thresholding. This order matters: thresholding raw SAR would convert random speckle into false water pixels.

### Required Result Table

| Item                               | Result                                                                       |
|:-----------------------------------|:-----------------------------------------------------------------------------|
| Water pixels                       | 1,038                                                                        |
| Flood-like water area              | 0.934 km2                                                                    |
| Mean backscatter in SAR flood zone | -19.62 dB                                                                    |
| Operational statement              | SAR detected 0.934 km2 of flood-like water under 100.0% optical cloud cover. |

![Task 1 SAR Detection](week10_outputs/task1_sar_detection_panel.png)

### How to Read Each Subplot

| Subplot                 | What it shows                                     | How to read it                                                                                                                               |
|:------------------------|:--------------------------------------------------|:---------------------------------------------------------------------------------------------------------------------------------------------|
| (a) Raw SAR VV          | Original Sentinel-1 VV dB image before denoising. | Dark pixels have low radar return. Water can be dark, but radar shadow and speckle can also be dark, so this panel is not a final flood map. |
| (b) Median filtered SAR | 5 x 5 median filter result.                       | This is the analysis image used for thresholding. The filter suppresses isolated speckle while keeping broader low-backscatter patches.      |
| (c) Binary flood mask   | Pixels where filtered VV < -18 dB after cleanup.  | This is the SAR flood-like water candidate mask. It is evidence, not ground truth.                                                           |
| (d) SAR plus overlay    | Flood mask over the filtered SAR image.           | This checks whether detected water is spatially coherent and whether it sits on plausible low-backscatter areas.                             |

### Result Interpretation

The final SAR flood-like mask contains **1,038 pixels**, or **0.934 km2**. The mean flood-zone VV value is **-19.62 dB**, which is consistent with low-backscatter water. However, this mask should not be interpreted as confirmed inundation by itself. SAR shadow, smooth wet soil, and steep terrain can also produce low backscatter. That is why the later fusion and topographic audit steps are necessary.


## Task 2: Sensor Fusion - Multi-Source Confidence Map

### How This Result Was Generated

The fusion map combines three binary sources: SAR water, NDWI optical water, and the Sentinel-2 SCL cloud mask. Because the selected optical scene is **100.0% cloud-covered**, NDWI cannot provide reliable confirmation. The rule therefore routes SAR detections in cloudy pixels into the SAR Only (Cloudy) class rather than calling them high confidence.

### Fusion Rule Table

| Optical NDWI water | SAR water | Cloud masked | Class |
|---|---|---|---|
| Yes | Yes | No | High Confidence |
| No or unusable | Yes | Yes | SAR Only (Cloudy) |
| Yes | No | No | Optical Only |
| No | No | Any | No Detection |

### Area Statistics

| class             |   code |   pixels |   area_km2 |
|:------------------|-------:|---------:|-----------:|
| No Detection      |      0 |   616810 |    555.129 |
| Optical Only      |      1 |        0 |      0     |
| SAR Only (Cloudy) |      2 |     1038 |      0.934 |
| High Confidence   |      3 |        0 |      0     |

![Task 2 Confidence Map](week10_outputs/task2_confidence_map.png)

### How to Read the Map

| Class               | Map color       | Meaning                                                                     | Action                                                                   |
|:--------------------|:----------------|:----------------------------------------------------------------------------|:-------------------------------------------------------------------------|
| 0 No Detection      | Gray/background | Neither SAR nor usable optical evidence detects water.                      | No immediate flood signal in this workflow.                              |
| 1 Optical Only      | Blue            | NDWI detects water but SAR does not, and the pixel is not cloud-masked.     | Needs manual review because one sensor disagrees.                        |
| 2 SAR Only (Cloudy) | Orange          | SAR detects flood-like water where optical evidence is clouded or unusable. | Main operational output in this case. Send field/UAV/gauge verification. |
| 3 High Confidence   | Red             | SAR and non-cloudy NDWI both detect water.                                  | Strongest evidence, but this run has 0 km2 because cloud cover is 100%.  |

### Result Interpretation

The most important result is not that High Confidence is zero; that is the correct outcome under complete cloud cover. The important result is that SAR still detects **0.934 km2** of flood-like water where optical data are unusable. Operationally, this map says: do not claim optical confirmation, but do not ignore the SAR signal. Treat orange SAR-only areas as reconnaissance targets and combine them with roads, settlements, gauges, and field reports.


## Task 3: Topographic Audit - DEM and Slope Assessment

### How This Result Was Generated

SAR is side-looking. In steep terrain, radar shadow, layover, and geometric distortion can create dark pixels that resemble water. To audit that risk, I loaded Copernicus DEM, derived slope, and flagged flood detections where slope is greater than **25 degrees**.

### False Positives Flagged by Slope Class

| slope_class   |   removed_pixels |   removed_km2 |
|:--------------|-----------------:|--------------:|
| 25-35 deg     |              104 |         0.094 |
| 35-45 deg     |              108 |         0.097 |
| >45 deg       |              469 |         0.422 |

![Task 3 Topographic Audit](week10_outputs/task3_topographic_audit.png)

### How to Read Each Panel

| Panel                                | What it shows                                                             | How to read it                                                                                                                     |
|:-------------------------------------|:--------------------------------------------------------------------------|:-----------------------------------------------------------------------------------------------------------------------------------|
| Left panel: fusion before audit      | Shows the SAR/optical fusion result before terrain screening.             | This is the raw emergency detection layer. It still contains possible SAR artifacts on steep terrain.                              |
| Middle panel: slope map              | Slope derived from Copernicus DEM.                                        | Bright or high-slope areas are places where standing floodwater is physically unlikely and SAR geometry artifacts are more likely. |
| Right panel: after topographic audit | Fusion map after marking slope > 25 degrees as false-positive candidates. | This separates the 0.613 km2 steep-slope warning area from the lower-slope post-audit candidate area of about 0.321 km2.           |

### Result Interpretation

The topographic audit flags **0.613 km2** of the SAR detections as steep-slope false-positive candidates. The remaining post-audit candidate area is about **0.321 km2**. This does not mean the flagged pixels are definitely wrong; it means they are physically suspicious because standing water is unlikely on steep slopes. These pixels should be reviewed with same-orbit SAR differencing, field reports, UAV imagery, or a newer DEM if the terrain changed during the disaster.


## AI Strategic Briefing

### Exact Prompt

```text
You are an emergency management advisor for Hualien County during Typhoon Fung-wong.
Based on these ARIA v7.0 sensor fusion results, generate a strategic operational briefing that covers:
1. Which areas require immediate evacuation?
2. How should resources be allocated between high-confidence and SAR-only zones?
3. What are the limitations of the current assessment?
4. What additional data would improve confidence?

Metrics:
- High confidence flood area: 0.000 km2
- SAR-only cloudy flood area: 0.934 km2
- False positives removed by topographic filter: 0.613 km2
- Cloud cover percentage: 100.0%
- SAR threshold: -18 dB, chosen as the ARIA default for conservative SAR flood extraction
- NDWI threshold: 0.0, chosen for turbid storm water rather than clear-water NDWI ~0.3
```

### LLM Response

Because the selected optical scene is 100.0% cloud-covered, there are no dual-sensor high-confidence flood pixels in this run. The operational priority should therefore be rapid field verification of the 0.934 km2 SAR-only cloudy zone rather than automatic evacuation based on optical confirmation.

For resource allocation, dispatch reconnaissance teams, UAVs, road patrols, and gauge checks first to SAR-only clusters near settlements, roads, bridges, and low-lying drainage corridors. Heavy evacuation or rescue assets should be staged nearby but committed after field reports or follow-up SAR/optical passes confirm persistent water.

The main limitations are SAR speckle, threshold sensitivity, radar shadow or layover on steep terrain, and the absence of optical confirmation under complete cloud cover. The slope audit flagged 0.613 km2 as likely steep-terrain false positives, leaving about 0.321 km2 as post-audit flood candidates.

Confidence would improve with river gauge records, disaster reports, UAV imagery, road closure data, settlement and road overlays, and a second Sentinel-1 acquisition from the same orbit.

### Reflection

The response is useful because it does not pretend that zero high-confidence area means zero flood risk. It correctly treats SAR-only detections as a reconnaissance priority under complete cloud cover. Its weakness is that the summary metrics do not contain village names, road segments, or population exposure, so the briefing cannot specify exact evacuation sites. I would use it as an incident-command triage memo and then overlay the map with roads, settlements, shelters, and live field reports.

## ARIA v7.0 vs. v6.0 Comparison

| Metric                           | W9 Optical Only              | W10 Fused                     | Improvement               |
|:---------------------------------|:-----------------------------|:------------------------------|:--------------------------|
| Total detected flood/change area | 33.346 km2                   | 0.934 km2                     | -32.412 km2               |
| Cloud-covered area analyzed      | 0 km2                        | 0.934 km2 SAR-only class      | cloud gaps audited by SAR |
| False positives handled          | phantom water removed by SCL | 0.613 km2 flagged by slope    | adds terrain audit        |
| Confidence levels                | 3-zone                       | 4-class + false-positive flag | finer triage              |

### Interpretation

W9 optical-only analysis mapped a broader disaster-change signal, including vegetation loss and debris-related spectral change. W10 is intentionally narrower: it asks whether SAR can detect flood-like water when the optical scene is clouded out. Therefore the decrease from W9's 33.346 km2 to W10's 0.934 km2 is not a performance failure; it reflects a stricter water-focused target and a more conservative SAR threshold.

The main improvement is operational rather than simply numerical: ARIA v7.0 can still produce an auditable flood candidate layer under 100.0% cloud cover. The topographic audit further separates physically plausible low-slope water from steep-terrain SAR artifacts.


## Figure-to-Output Traceability

| Output file | Produced from | What it supports |
|---|---|---|
| `task1_sar_detection_panel.png` | SAR dB, median filter, SAR threshold mask | Task 1 SAR flood extraction and area calculation |
| `task2_confidence_map.png` | SAR mask + NDWI mask + cloud mask | Task 2 four-class confidence interpretation |
| `task3_topographic_audit.png` | Fusion map + DEM-derived slope | Task 3 false-positive screening |
| `summary_metrics.csv` | Final computed metrics | Executive summary and Task 1 table |
| `task2_fusion_area_stats.csv` | Pixel counts by fusion class | Confidence-map area statistics |
| `task3_false_positive_by_slope.csv` | Slope classes over flood detections | Topographic audit quantification |
| `task4_ai_briefing_and_report.md` | Summary metrics and comparison table | AI strategic briefing and ARIA v6/v7 comparison |

This traceability table is included so that each figure can be tied back to a processing step and a deliverable requirement.


## Output Verification and Sanity Checks

| Check | Result |
|---|---|
| Median filter before thresholding | Passed |
| SAR threshold documented | `-18 dB` |
| Flood mask covers only a small part of the BBOX | Passed: 0.934 km2 out of the analysis grid |
| Cloud-cover scenario matches assignment | Passed: 100.0% cloud cover |
| Fusion logic handles clouded optical data | Passed: high-confidence is 0, SAR-only cloudy is 0.934 km2 |
| Topographic audit performed | Passed: 0.613 km2 flagged |
| Output files saved | Figures, CSV tables, briefing markdown, and notebook are under `week10_outputs/` |

### Final Interpretation

ARIA v7.0 adds value precisely when optical satellites fail. In this run, W10 does not claim dual-sensor confirmation because the optical scene is cloud-covered. Instead, it produces a conservative SAR-only candidate layer and then uses slope to identify likely terrain artifacts. That is a more honest emergency product than forcing a high-confidence label where optical evidence does not exist.


## Submission Checklist

- [x] SAR flood extraction with median filtering
- [x] 2 x 2 SAR visualization panel
- [x] Flood area, pixel count, and mean backscatter table
- [x] Four-class fusion map and area statistics
- [x] Per-figure explanation and traceability
- [x] DEM/slope topographic audit
- [x] False-positive table by slope class
- [x] AI strategic briefing with exact prompt, response, and reflection
- [x] W9 vs W10 comparison table
- [x] `.env.example` for reproducibility
- [x] Discussion and output verification included
